IMPORT LIBRARIES

## 📌 Project summary

This notebook implements a small feedforward neural network written from scratch using NumPy. It's trained on synthetic 2D data (scikit-learn's make_blobs). The goal is pedagogical: to understand initialization, forward propagation, backpropagation, parameter updates, and to visualize the decision boundary evolving during training.

- Libraries used: NumPy, Matplotlib, Scikit-learn
- Dynamic visualization of the decision boundary
- Typical accuracy obtained on the example dataset: ~90%+ depending on random seed and hyperparameters


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.metrics import accuracy_score
from IPython.display import display, clear_output
import time

# Optional: set a random seed for reproducibility
np.random.seed(0)

INITIALIZATION FUNCTION

In [ ]:
def initialize_parameters(n0, n1, n2):
    """
    Initialize parameters for a 2-layer network (input -> hidden -> output).
    n0: input dimension
    n1: hidden units
    n2: output units (for binary classification n2=1)
    """
    W1 = np.random.randn(n1, n0) * 0.01
    b1 = np.zeros((n1, 1))
    W2 = np.random.randn(n2, n1) * 0.01
    b2 = np.zeros((n2, 1))

    parameters = {
        "W1": W1,
        "b1": b1,
        "W2": W2,
        "b2": b2
    }
    return parameters

FORWARD PROPAGATION

In [ ]:
def forward_propagation(X, parameters):
    """
    Forward propagation for a 2-layer network with sigmoid activations.
    X: input of shape (n_x, m)
    returns activations dict with A1 and A2
    """
    W1 = parameters["W1"]
    b1 = parameters["b1"]
    W2 = parameters["W2"]
    b2 = parameters["b2"]

    Z1 = W1.dot(X) + b1
    A1 = 1 / (1 + np.exp(-Z1))  # sigmoid
    Z2 = W2.dot(A1) + b2
    A2 = 1 / (1 + np.exp(-Z2))  # sigmoid (binary output)

    activations = {
        "A1": A1,
        "A2": A2,
        "Z1": Z1,
        "Z2": Z2
    }
    return activations

BACKWARD PROPAGATION

In [ ]:
def backward_propagation(X, y, parameters, activations):
    """
    Backpropagation for the 2-layer network using binary cross-entropy loss and sigmoid activations.
    X: (n_x, m)
    y: (1, m)
    """
    A1 = activations["A1"]
    A2 = activations["A2"]
    W2 = parameters["W2"]

    m = y.shape[1]
    dZ2 = A2 - y  # derivative of BCE with sigmoid output
    dW2 = (1.0 / m) * dZ2.dot(A1.T)
    db2 = (1.0 / m) * np.sum(dZ2, axis=1, keepdims=True)
    # derivative through sigmoid for hidden layer
    dZ1 = W2.T.dot(dZ2) * (A1 * (1 - A1))
    dW1 = (1.0 / m) * dZ1.dot(X.T)
    db1 = (1.0 / m) * np.sum(dZ1, axis=1, keepdims=True)

    gradients = {
        "dW1": dW1,
        "dW2": dW2,
        "db1": db1,
        "db2": db2
    }
    return gradients

PARAMETER UPDATE

In [ ]:
def update_parameters(gradients, parameters, learning_rate):
    """
    Update parameters in-place using simple gradient descent.
    """

    parameters["W1"] -= learning_rate * gradients["dW1"]
    parameters["b1"] -= learning_rate * gradients["db1"]
    parameters["W2"] -= learning_rate * gradients["dW2"]
    parameters["b2"] -= learning_rate * gradients["db2"]

    return parameters

LOG LOSS

In [ ]:
def log_loss(A, y):
    """
    Binary cross-entropy loss. A and y should have the same shape (1, m).
    """
    if y.shape[1] == 0:
        return 0.0
    epsilon = 1e-15
    return -np.mean(y * np.log(A + epsilon) + (1 - y) * np.log(1 - A + epsilon))

PREDICTION FUNCTION

In [ ]:
def predict(X, parameters):
    activations = forward_propagation(X, parameters)
    A2 = activations["A2"]
    return (A2 >= 0.5).astype(int)



NETWORK TRAINING FUNCTION

In [ ]:
def neural_network(X_train, y_train, learning_rate=0.1, n_iter=200, n1=16):
    """
    Train a simple 2-layer neural network and display training curves and decision boundary.
    X_train: (n_x, m)
    y_train: (1, m)
    """

    n0 = X_train.shape[0]
    n2 = y_train.shape[0]
    parameters = initialize_parameters(n0, n1, n2)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    plt.ion()
    fig.suptitle("Training progress")
    line_loss, = ax1.plot([], [], 'r-', label='Loss')
    line_acc, = ax2.plot([], [], 'b-', label='Accuracy')

    for ax in (ax1, ax2):
        ax.set_xlabel('Iterations (x10)')
        ax.grid(True, linestyle='--', alpha=0.6)
        ax.legend()

    ax1.set_ylabel('Loss')
    ax2.set_ylabel('Accuracy')

    train_loss = []
    train_acc = []

    for i in range(n_iter):
        # Forward and backward propagation
        activations = forward_propagation(X_train, parameters)
        gradients = backward_propagation(X_train, y_train, parameters, activations)
        parameters = update_parameters(gradients, parameters, learning_rate)

        # Recompute activations after update so metrics reflect the updated model
        activations = forward_propagation(X_train, parameters)

        if i % 10 == 0:
            loss = log_loss(activations["A2"], y_train)
            y_pred = (activations["A2"] >= 0.5).astype(int)
            acc = accuracy_score(y_train.flatten(), y_pred.flatten())

            train_loss.append(loss)
            train_acc.append(acc)

            line_loss.set_data(range(len(train_loss)), train_loss)
            line_acc.set_data(range(len(train_acc)), train_acc)

            ax1.relim()
            ax1.autoscale_view()
            ax2.set_ylim(0, 1.05)
            ax2.set_xlim(0, max(1, len(train_acc)))

            fig.canvas.draw()
            fig.canvas.flush_events()
            time.sleep(0.01)

    # Final metrics (after training)
    final_activations = forward_propagation(X_train, parameters)
    final_pred = (final_activations["A2"] >= 0.5).astype(int)
    final_acc = accuracy_score(y_train.flatten(), final_pred.flatten())
    print("\033[32mAccuracy:", final_acc, "\033[0m")
    plt.ioff()
    plt.show()

    # Decision boundary visualization
    x_min, x_max = X_train[0, :].min() - 1, X_train[0, :].max() + 1
    y_min, y_max = X_train[1, :].min() - 1, X_train[1, :].max() + 1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                         np.linspace(y_min, y_max, 200))

    grid_points = np.c_[xx.ravel(), yy.ravel()].T  # shape (2, num_points)
    grid_activations = forward_propagation(grid_points, parameters)
    Z = grid_activations["A2"]
    Z = Z.reshape(xx.shape)

    plt.figure(figsize=(8, 6))
    plt.contourf(xx, yy, Z, levels=[0, 0.5, 1], cmap=plt.cm.Spectral, alpha=0.4)
    plt.scatter(X_train[0, :], X_train[1, :], c=y_train.flatten(), cmap=plt.cm.Spectral, edgecolors='k')
    plt.title("Neural Network Decision Boundary")

    plt.xlabel("x1")

    plt.ylabel("x2")

    plt.show()

    return parameters

EXECUTION AND RESULTS

In [ ]:
X, y = make_blobs(n_samples=200, n_features=2, centers=2, random_state=0)
X = X.T
y = y.reshape((1, y.shape[0]))
parameters = neural_network(X, y, learning_rate=0.5, n_iter=200, n1=8)